# 00 — Audit existing segmentation runtime (NO training)

Attach exactly one extracted 02 output. GPU + Internet, then Run All. Reproduces full validation and compares 200 fixed-seed scenes at the source label grid using old versus training-matched input size. Downloads official LoveDA automatically. Exports real masks, previews and paired-scene intervals. This never changes the existing model or its release gate. Setup may require one kernel restart.

In [ ]:
from pathlib import Path
import json
roots = [p.parent for p in Path('/kaggle/input').rglob('training_manifest.json') if (p.parent / 'model.safetensors').is_file()]
if len(roots) != 1: raise RuntimeError('Attach exactly one extracted notebook 02 output.')
AUDIT_CROP = json.loads((roots[0] / 'training_manifest.json').read_text())['config']['crop_size']
if type(AUDIT_CROP) is not int or not 128 <= AUDIT_CROP <= 1024: raise ValueError('Invalid recorded crop size')


# SatQuery semantic-mask specialist — free Kaggle training

This notebook trains a **pixel-supervised** land-cover model for the single-image overlay.
It does not pretend that additional Qwen VQA examples can teach exact pixel boundaries.

The default free-T4 profile uses 70% of LoveDA's official **training split**, stratified by
urban/rural domain. Validation is excluded from training and used for checkpoint selection. Increase the
fraction only after the default run completes; never mix validation/test images into training.

LoveDA is academic/non-commercial and derived from Google Earth imagery. It is suitable for an
SIH research prototype, not automatically for a commercial release. The first transfer model
must still be evaluated on representative Indian/ISRO imagery before operational claims.

## 0. Install a pinned cloud environment

In [ ]:
import subprocess
import sys
import os
from importlib import metadata

# Set before importing torch: use one GPU, not notebook DataParallel on two T4s.
if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("Start a fresh GPU session before running this memory-safe notebook.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PACKAGES = [
    "numpy==2.2.6",
    "scipy==1.15.3",
    "opencv-python-headless==4.11.0.86",
    "albucore==0.0.24",
    "transformers==4.57.1",
    "accelerate==1.7.0",
    "huggingface_hub==0.36.2",
    "albumentations==2.0.8",
    "safetensors>=0.5,<1",
    "tensorboard>=2.18,<3",
    "requests>=2.32,<3",
    "tqdm>=4.66,<5",
]
def installed_training_versions():
    return {d.metadata["Name"].lower().replace("_", "-"): d.version for d in metadata.distributions() if d.metadata["Name"]}

before_install = installed_training_versions()
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *PACKAGES])
after_install = installed_training_versions()
numeric_probe = "import numpy, numpy.testing; from scipy import special; import albumentations; from transformers import SegformerForSemanticSegmentation"
probe = subprocess.run([sys.executable, "-c", numeric_probe], capture_output=True, text=True)
if probe.returncode:
    raise RuntimeError("Fresh-process segmentation imports failed:\n" + probe.stderr[-6000:])
if any(before_install.get(name) != after_install.get(name) for name in ("numpy", "scipy", "transformers", "huggingface-hub", "albumentations")):
    raise SystemExit("SETUP COMPLETE — RESTART KERNEL (keep session files), then Run All again. Do not hot-reload NumPy.")
try:
    exec(numeric_probe)
except Exception as exc:
    raise RuntimeError("Disk imports pass but kernel is stale. Restart Kernel, then Run All.") from exc
print("PASS: segmentation numeric/import environment is ready.")

## 1. Configuration — edit only this cell

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import os


@dataclass(frozen=True)
class Config:
    base_model: str = "nvidia/mit-b0"
    base_revision: str = "80983a413c30d36a39c20203974ae7807835e2b4"
    output_repo: str = "aanandmodi/satquery-segformer-loveda"
    train_fraction: float = 0.70
    seed: int = 42
    crop_size: int = AUDIT_CROP
    epochs: int = 20
    training_revision: str = "runtime-audit-no-training"
    train_batch_size: int = 1
    eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 6e-5
    warm_start_learning_rate: float = 2e-5
    weight_decay: float = 0.01
    dice_weight: float = 0.5
    num_workers: int = 0
    push_to_hub: bool = False
    make_repo_private: bool = True


CFG = Config()
assert 0 < CFG.train_fraction <= 1
ROOT = Path(
    "/kaggle/working/satquery-segmentation-r3"
    if Path("/kaggle/working").exists()
    else "/content/satquery-segmentation-r3"
)
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "best-model"
EVAL_DIR = ROOT / "evaluation"
for directory in (DATA_DIR, OUTPUT_DIR, EVAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(ROOT / "hf-cache"))
print(asdict(CFG))

## 2. GPU, seeds and optional Hugging Face token

In [ ]:
import getpass
import json
import random
import numpy as np
import torch

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle/Colab GPU before continuing.")
torch.backends.cuda.matmul.allow_tf32 = True


def optional_secret(name: str) -> str | None:
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient

        return (UserSecretsClient().get_secret(name) or "").strip() or None
    except Exception:
        return None


HF_TOKEN = optional_secret("HF_TOKEN")
if CFG.push_to_hub and not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HF write token (hidden): ").strip()
    if not HF_TOKEN:
        raise RuntimeError("push_to_hub=True requires an HF write token.")
print({"gpu": torch.cuda.get_device_name(0), "push_to_hub": CFG.push_to_hub})

## 3. Download the official LoveDA train and validation splits

The archives and MD5 values are from the dataset publisher's release. The public test masks are
unavailable, so this notebook reports development metrics on the official validation split and
never calls them test metrics. Download is several GB; enable Internet in Kaggle.

In [ ]:
import hashlib
import zipfile
import requests
from PIL import Image
from tqdm.auto import tqdm

ARCHIVES = {
    "train": {
        "url": "https://zenodo.org/records/5706578/files/Train.zip?download=1",
        "filename": "Train.zip",
        "md5": "de2b196043ed9b4af1690b3f9a7d558f",
    },
    "val": {
        "url": "https://zenodo.org/records/5706578/files/Val.zip?download=1",
        "filename": "Val.zip",
        "md5": "84cae2577468ff0b5386758bb386d31d",
    },
}


def file_md5(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.md5()  # nosec B324 -- publisher checksum, not a security primitive
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def download_and_extract(split: str) -> None:
    metadata = ARCHIVES[split]
    destination = DATA_DIR / metadata["filename"]
    extracted = DATA_DIR / split.capitalize()
    if not destination.exists() or file_md5(destination) != metadata["md5"]:
        response = requests.get(metadata["url"], stream=True, timeout=(30, 300))
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with destination.open("wb") as handle, tqdm(
            total=total, unit="B", unit_scale=True, desc=metadata["filename"]
        ) as progress:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    handle.write(chunk)
                    progress.update(len(chunk))
    if file_md5(destination) != metadata["md5"]:
        raise RuntimeError(f"Publisher checksum failed for {destination.name}")
    if not extracted.exists():
        with zipfile.ZipFile(destination) as archive:
            archive.extractall(DATA_DIR)


class LoveDASource:
    def __init__(self, split: str):
        root = DATA_DIR / split.capitalize()
        images = sorted(root.glob("*/images_png/*.png"))
        self.files = [
            {"image": str(path), "mask": str(path).replace("images_png", "masks_png")}
            for path in images
        ]
        if not self.files or any(not Path(item["mask"]).exists() for item in self.files):
            raise RuntimeError(f"Incomplete LoveDA {split} extraction at {root}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        item = self.files[index]
        with Image.open(item["image"]) as image:
            pixels = torch.from_numpy(np.array(image.convert("RGB"), copy=True)).permute(2, 0, 1)
        with Image.open(item["mask"]) as mask:
            labels = torch.from_numpy(np.array(mask, copy=True))
        return {"image": pixels, "mask": labels}


for split_name in ARCHIVES:
    download_and_extract(split_name)
train_source = LoveDASource("train")
validation_source = LoveDASource("val")
print({"official_train": len(train_source), "official_validation": len(validation_source)})

## 4. Select 70% of train without touching validation

Sampling is deterministic and stratified by LoveDA's urban/rural folders. The selected path
hashes are written to the manifest, making the run reproducible and auditable.

In [ ]:
def domain_for(item: dict[str, str]) -> str:
    normalized = item["image"].replace("\\", "/").lower()
    return "urban" if "/urban/" in normalized else "rural"


def stratified_indexes(source, fraction: float, seed: int) -> list[int]:
    rng = random.Random(seed)
    groups: dict[str, list[int]] = {"urban": [], "rural": []}
    for index, item in enumerate(source.files):
        groups[domain_for(item)].append(index)
    selected: list[int] = []
    for indexes in groups.values():
        rng.shuffle(indexes)
        selected.extend(indexes[: max(1, round(len(indexes) * fraction))])
    return sorted(selected)


train_indexes = stratified_indexes(train_source, CFG.train_fraction, CFG.seed)
train_names = [Path(train_source.files[index]["image"]).as_posix() for index in train_indexes]
selection_sha256 = hashlib.sha256("\n".join(train_names).encode()).hexdigest()
print({
    "selected_train": len(train_indexes),
    "train_fraction": len(train_indexes) / len(train_source),
    "selection_sha256": selection_sha256,
})

## 5. Pixel-safe augmentation and datasets

LoveDA labels are 1–7 with no-data 0. We remap them to model labels 0–6 and no-data 255. Image
transforms use bilinear interpolation while masks use nearest-neighbour interpolation.

In [ ]:
import albumentations as A
from torch.utils.data import Dataset
from transformers import SegformerImageProcessor

LABELS = ["background", "building", "road", "water", "barren", "forest", "agricultural"]
ID2LABEL = {index: label for index, label in enumerate(LABELS)}
LABEL2ID = {label: index for index, label in ID2LABEL.items()}

train_transform = A.Compose([
    # Match validation's full-scene scale; the old native 384px crop trained at
    # ~2.7x the evaluation magnification. Random rescaling supplies nearby scales.
    A.Resize(height=CFG.crop_size, width=CFG.crop_size),
    A.RandomScale(scale_limit=(-0.15, 0.35), p=0.75),
    A.PadIfNeeded(min_height=CFG.crop_size, min_width=CFG.crop_size, border_mode=0, fill=0, fill_mask=0),
    A.RandomCrop(height=CFG.crop_size, width=CFG.crop_size),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.03, p=0.35),
])
validation_transform = A.Compose([
    A.Resize(height=CFG.crop_size, width=CFG.crop_size),
])
processor = SegformerImageProcessor(
    do_resize=False,
    do_reduce_labels=False,
)


class LoveDASegmentationDataset(Dataset):
    def __init__(self, source, indexes, transform):
        self.source = source
        self.indexes = list(indexes)
        self.transform = transform

    def __len__(self):
        return len(self.indexes)

    def __getitem__(self, position):
        source_index = self.indexes[position]
        sample = self.source[source_index]
        image = sample["image"].permute(1, 2, 0).byte().numpy()
        mask = sample["mask"].numpy().astype(np.uint8)
        augmented = self.transform(image=image, mask=mask)
        remapped = np.where(augmented["mask"] == 0, 255, augmented["mask"] - 1).astype(np.uint8)
        encoded = processor(
            images=augmented["image"],
            segmentation_maps=remapped,
            return_tensors="pt",
        )
        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": encoded["labels"].squeeze(0).long(),
            "source_index": torch.tensor(source_index),
        }


train_dataset = LoveDASegmentationDataset(
    train_source, train_indexes, train_transform
)
validation_dataset = LoveDASegmentationDataset(
    validation_source, range(len(validation_source)), validation_transform
)
print({"train": len(train_dataset), "validation": len(validation_dataset), "labels": LABELS})

## 6. Estimate class weights from training masks only

In [ ]:
class_counts = np.zeros(len(LABELS), dtype=np.int64)
for position, index in enumerate(train_indexes):
    with Image.open(train_source.files[index]["mask"]) as source_mask:
        mask = np.asarray(source_mask)
    for publisher_id in range(1, 8):
        class_counts[publisher_id - 1] += int((mask == publisher_id).sum())
    if (position + 1) % 500 == 0:
        print("scanned", position + 1)
frequencies = class_counts / class_counts.sum()
class_weights = 1 / np.log(1.02 + frequencies)
class_weights = class_weights / class_weights.mean()
print({label: round(float(class_weights[index]), 4) for index, label in ID2LABEL.items()})

## 7. Load SegFormer and define weighted cross-entropy + Dice

In [ ]:
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation, Trainer

model = SegformerForSemanticSegmentation.from_pretrained(
    CFG.base_model,
    revision=CFG.base_revision,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    num_labels=len(LABELS),
    ignore_mismatched_sizes=True,
    trust_remote_code=False,
)
weight_tensor = torch.tensor(class_weights, dtype=torch.float32)

# Optional warm start: attach the preserved 02 output (extracted, not ZIP-only).
# Load ONLY weights; do not import its optimizer, failed gate or old training schedule.
warm_candidates = [p.parent for p in Path("/kaggle/input").rglob("training_manifest.json")
                   if (p.parent / "model.safetensors").is_file()]
if len(warm_candidates) > 1:
    raise RuntimeError("Attach at most one previous 02 output for warm start.")
warm_start = None
if warm_candidates:
    from safetensors.torch import load_file
    old_root = warm_candidates[0]
    old_manifest = json.loads((old_root / "training_manifest.json").read_text())
    old_hashes = json.loads((old_root / "sha256_manifest.json").read_text())
    weight_sha = hashlib.sha256((old_root / "model.safetensors").read_bytes()).hexdigest()
    for name in ("model.safetensors", "config.json", "training_manifest.json", "preprocessor_config.json"):
        if old_hashes.get(name) != hashlib.sha256((old_root / name).read_bytes()).hexdigest():
            raise RuntimeError(f"Previous 02 checksum mismatch: {name}")
    old_config = json.loads((old_root / "config.json").read_text())
    if {int(k): v for k, v in old_config["id2label"].items()} != ID2LABEL:
        raise RuntimeError("Previous 02 label order differs; refuse unsafe warm start.")
    state = load_file(old_root / "model.safetensors")
    if not all(torch.isfinite(t).all() for t in state.values()):
        raise RuntimeError("Previous 02 contains non-finite weights.")
    model.load_state_dict(state, strict=True)
    warm_start = {"weights_sha256": weight_sha, "source": str(old_root)}
    del state
print({"warm_start": warm_start, "revision": CFG.training_revision})


class DiceCETrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # This custom mean loss does not use num_items_in_batch; Trainer must divide
        # by gradient_accumulation_steps rather than treating it as pre-normalized.
        self.model_accepts_loss_kwargs = False

    def create_optimizer(self):
        if self.optimizer is None:
            encoder, decoder = [], []
            for name, parameter in self.model.named_parameters():
                if parameter.requires_grad:
                    (decoder if name.startswith("decode_head.") else encoder).append(parameter)
            self.optimizer = torch.optim.AdamW([
                {"params": encoder, "lr": self.args.learning_rate},
                {"params": decoder, "lr": self.args.learning_rate * 5},
            ], weight_decay=self.args.weight_decay)
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        inputs.pop("source_index", None)
        outputs = model(**inputs)
        logits = F.interpolate(
            outputs.logits, size=labels.shape[-2:], mode="bilinear", align_corners=False
        ).float()
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Non-finite segmentation logits; no release is permitted.")
        valid = labels != 255
        if not valid.any():
            loss = logits.sum() * 0.0
            return (loss, outputs) if return_outputs else loss
        ce = F.cross_entropy(
            logits, labels, weight=weight_tensor.to(logits.device), ignore_index=255
        )
        # No full int64 one-hot mask; float32 reductions avoid fp16 overflow.
        probabilities = logits.float().softmax(dim=1)
        dice_terms = []
        for class_id in range(len(LABELS)):
            expected = labels == class_id
            if expected.any():
                probability = probabilities[:, class_id] * valid
                intersection = (probability * expected).sum()
                denominator = probability.sum() + expected.sum()
                dice_terms.append((2 * intersection + 1) / (denominator + 1))
        dice_loss = 1 - torch.stack(dice_terms).mean()
        loss = ce + CFG.dice_weight * dice_loss
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite segmentation loss; preserve diagnostics, stop training.")
        return (loss, outputs) if return_outputs else loss


print({"parameters": sum(parameter.numel() for parameter in model.parameters())})

## 8. Real validation metrics

Streaming confusion matrix: never retain full-validation masks/labels. Moving accumulated
masks to CPU with eval_accumulation_steps alone still exhausts system RAM.

In [ ]:
from transformers import EvalPrediction


def preprocess_logits(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = F.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
    return logits.argmax(dim=1).to(torch.uint8)


def update_confusion(confusion, predictions, references):
    valid = (references >= 0) & (references < len(LABELS))
    encoded = len(LABELS) * references[valid].astype(np.int64) + predictions[valid]
    confusion += np.bincount(encoded, minlength=len(LABELS) ** 2).reshape(confusion.shape)


def metrics_from_confusion(confusion) -> dict[str, float]:
    metrics: dict[str, float] = {}
    ious, dices = [], []
    for class_id, label in ID2LABEL.items():
        intersection = confusion[class_id, class_id]
        denominator = confusion[:, class_id].sum() + confusion[class_id, :].sum()
        union = denominator - intersection
        iou = float(intersection / union) if union else float("nan")
        dice = float(2 * intersection / denominator) if denominator else float("nan")
        metrics[f"iou_{label}"] = iou
        metrics[f"dice_{label}"] = dice
        if np.isfinite(iou):
            ious.append(iou)
        if np.isfinite(dice):
            dices.append(dice)
    metrics["mean_iou"] = float(np.mean(ious)) if ious else 0.0
    metrics["mean_dice"] = float(np.mean(dices)) if dices else 0.0
    metrics["pixel_accuracy"] = float(np.trace(confusion) / max(1, confusion.sum()))
    # Checkpoint selection balances all gates; accuracy thresholds are unchanged.
    if all(f"iou_{name}" in metrics for name in ("water", "forest", "agricultural")):
        values = [metrics["mean_iou"] / 0.45] + [metrics[f"iou_{name}"] / 0.35 for name in ("water", "forest", "agricultural")]
        metrics["release_balance"] = min(values) if all(np.isfinite(values)) else 0.0
    return metrics


class StreamingSegmentationMetrics:
    def __init__(self):
        self.confusion = np.zeros((len(LABELS), len(LABELS)), dtype=np.int64)

    def __call__(self, evaluation: EvalPrediction, compute_result: bool = False):
        def cpu(value):
            return value.detach().cpu().numpy() if hasattr(value, "detach") else np.asarray(value)

        update_confusion(self.confusion, cpu(evaluation.predictions), cpu(evaluation.label_ids))
        if not compute_result:
            return {}
        result = metrics_from_confusion(self.confusion)
        self.confusion.fill(0)
        return result

## 9. Train with checkpoint/resume

In [ ]:
from transformers import EarlyStoppingCallback, TrainingArguments

training_args = TrainingArguments(
    output_dir=str(ROOT / "checkpoints"),
    learning_rate=CFG.warm_start_learning_rate if warm_start else CFG.learning_rate,
    weight_decay=CFG.weight_decay,
    num_train_epochs=CFG.epochs,
    per_device_train_batch_size=CFG.train_batch_size,
    per_device_eval_batch_size=CFG.eval_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=False,  # T4/P100 use FP32; avoid silent overflow in the new quality run.
    lr_scheduler_type="cosine",
    warmup_ratio=0.08,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="release_balance",
    greater_is_better=True,
    dataloader_num_workers=CFG.num_workers,
    dataloader_pin_memory=False,
    batch_eval_metrics=True,
    save_safetensors=True,
    eval_accumulation_steps=1,
    remove_unused_columns=False,
    report_to="none",
    seed=CFG.seed,
    data_seed=CFG.seed,
)
trainer = DiceCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=StreamingSegmentationMetrics(),
    preprocess_logits_for_metrics=preprocess_logits,
)


In [ ]:
"""Embedded in the numbered Kaggle notebooks; no repository checkout required."""

import base64
import csv
import hashlib
import io
import json
import shutil
import time
import urllib.parse
import urllib.request
from pathlib import Path


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def prepare_flood_data(destination):
    """Download only official hand-labelled triplets, with GCS generation/MD5 checks."""
    root = Path(destination)
    root.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(root).free < 5 * 1024**3:
        raise RuntimeError(
            "Keep at least 5 GiB free for the Sen1Floods11 data and checkpoints."
        )
    base = "https://storage.googleapis.com/sen1floods11/"
    origins = []

    def fetch(object_name, target):
        metadata_url = (
            "https://storage.googleapis.com/storage/v1/b/sen1floods11/o/"
            + urllib.parse.quote(object_name, safe="")
        )
        with urllib.request.urlopen(metadata_url, timeout=60) as response:
            metadata = json.load(response)
        expected = metadata["md5Hash"]

        def matches(path):
            if not path.is_file() or path.stat().st_size != int(metadata["size"]):
                return False
            return (
                base64.b64encode(hashlib.md5(path.read_bytes()).digest()).decode()
                == expected
            )

        target.parent.mkdir(parents=True, exist_ok=True)
        if not matches(target):
            temporary = target.with_suffix(target.suffix + ".partial")
            for attempt in range(3):
                try:
                    url = base + object_name + "?generation=" + metadata["generation"]
                    with (
                        urllib.request.urlopen(url, timeout=120) as response,
                        temporary.open("wb") as out,
                    ):
                        shutil.copyfileobj(response, out)
                    if not matches(temporary):
                        raise ValueError(f"Source checksum mismatch: {object_name}")
                    temporary.replace(target)
                    break
                except Exception:
                    if attempt == 2:
                        raise
                    time.sleep(2 * (attempt + 1))
        origins.append(
            {
                "object": object_name,
                "generation": metadata["generation"],
                "md5": expected,
                "sha256": file_sha256(target),
            }
        )

    chip_ids = set()
    for split in ("train", "valid", "test"):
        source = f"v1.1/splits/flood_handlabeled/flood_{split}_data.csv"
        csv_path = root / "splits" / f"flood_{split}_data.csv"
        fetch(source, csv_path)
        identifiers = []
        for row in csv.reader(io.StringIO(csv_path.read_text(encoding="utf-8"))):
            if not row:
                continue
            name = Path(row[0].strip()).name
            if not name.endswith("_S1Hand.tif"):
                raise ValueError(f"Unexpected official split entry: {row}")
            identifier = name.removesuffix("_S1Hand.tif")
            identifiers.append(identifier)
            chip_ids.add(identifier)
        (root / "splits" / f"flood_{split}_data.txt").write_text(
            "\n".join(identifiers) + "\n", encoding="utf-8"
        )
    for number, identifier in enumerate(sorted(chip_ids), 1):
        for remote, local, suffix in (
            ("S1Hand", "S1GRDHand", "S1Hand"),
            ("S2Hand", "S2L1CHand", "S2Hand"),
            ("LabelHand", "LabelHand", "LabelHand"),
        ):
            filename = f"{identifier}_{suffix}.tif"
            fetch(
                f"v1.1/data/flood_events/HandLabeled/{remote}/{filename}",
                root / "data" / local / filename,
            )
        if number % 20 == 0 or number == len(chip_ids):
            print(
                f"Verified Sen1Floods11 triplets: {number}/{len(chip_ids)}", flush=True
            )
    (root / "source_objects.json").write_text(
        json.dumps(origins, indent=2), encoding="utf-8"
    )
    return root


def find_trained_artifacts(input_root, *, allow_experimental_segmentation=False):
    """Verify weights AND the metadata that controls preprocessing/release decisions.

    Checksums establish internal consistency, not independent authorship or model accuracy.
    """
    candidates = {"segmentation": [], "change": [], "fusion": []}
    for path in Path(input_root).rglob("config.json"):
        root = path.parent
        if not (root / "model.safetensors").is_file():
            continue
        config = json.loads(path.read_text(encoding="utf-8"))
        architecture = config.get("architecture")
        if architecture == "shared_resnet18_gru_answer_mask":
            role = "change"
        elif architecture == "terramind_s1_s2_pixel_flood_segmentation":
            role = "fusion"
        elif (
            config.get("model_type") == "segformer"
            and (root / "training_manifest.json").is_file()
        ):
            role = "segmentation"
        else:
            continue
        manifest_path = root / "sha256_manifest.json"
        if not manifest_path.is_file():
            raise ValueError(f"Missing hash manifest in {root}")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        required = ["model.safetensors", "config.json"]
        required += (["training_manifest.json", "preprocessor_config.json"]
                     if role == "segmentation" else ["release_gate.json"])
        for name in required:
            if not (root / name).is_file() or manifest.get(name) != file_sha256(root / name):
                raise ValueError(f"Checkpoint integrity failed: {root / name}")
        if role == "segmentation":
            report = json.loads(
                (root / "training_manifest.json").read_text(encoding="utf-8")
            )
            candidate = report.get("release_candidate") is True
            if not candidate and not allow_experimental_segmentation:
                raise ValueError(
                    "SegFormer validation gate failed. Review its metrics before serving."
                )
            if not candidate:
                print("EXPERIMENTAL DEMO ONLY: SegFormer failed its release gate. "
                      "Its masks must remain labelled unvalidated; no release flag is changed.")
        else:
            gate_path = root / "release_gate.json"
            gate = (
                json.loads(gate_path.read_text(encoding="utf-8"))
                if gate_path.is_file()
                else {}
            )
            if gate.get("validation_gate_passed") is not True or gate.get("test_gate_passed") is not True:
                raise ValueError(
                    f"{role} needs passing validation/test gates from this numbered pack."
                )
        candidates[role].append(root)
    for role, roots in candidates.items():
        if len(roots) != 1:
            raise ValueError(
                f"Attach exactly one passing {role} output from notebooks 02/03/04. Found {len(roots)}. "
                "Use Kaggle Add Input → Notebook Output, or attach the extracted inference zip."
            )
    return {role: roots[0] for role, roots in candidates.items()}


def export_inference_zip(source, filename):
    import zipfile

    source, target = Path(source), Path(filename)
    with zipfile.ZipFile(target, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(source.rglob("*")):
            if path.is_file() and path.name != "training_state.pt":
                archive.write(path, Path(source.name) / path.relative_to(source))
    print(f"DOWNLOAD / PRESERVE: {target}", flush=True)
    return target


In [ ]:
def quality_preprocessing_contract(root):
    """Recover the actual validation resize, not the processor's unused size default."""
    if not root:
        return {'mode': 'published_processor', 'input_size': None}
    report = json.loads((Path(root) / 'training_manifest.json').read_text())
    size = report.get('config', {}).get('crop_size')
    if type(size) is not int or not 128 <= size <= 1024:
        raise ValueError('SegFormer export has no supported, recorded validation crop_size.')
    return {'mode': 'training_matched_full_scene', 'input_size': size, 'image_resize': 'opencv_linear', 'logit_resize': 'bilinear_align_corners_false', 'source': 'training_manifest.json:config.crop_size'}

@torch.inference_mode()
def quality_semantic_prediction(image):
    """Reuse scene probabilities across targets; optional bounded overlap tiles."""
    tiled = globals().get('QUALITY_TILED_SEGMENTATION', False)
    contract = globals().get('SEGMENTATION_PREPROCESSING', {'input_size': None})
    if tiled and contract.get('input_size'):
        raise ValueError('Tiled inference is not validated for this full-scene-trained checkpoint.')
    edge, stride = (512, 448)

    def starts(length):
        return sorted(set([*range(0, max(1, length - edge + 1), stride), max(0, length - edge)]))
    crops = [(0, 0, image.width, image.height)] if not tiled else [(x, y, min(x + edge, image.width), min(y + edge, image.height)) for y in starts(image.height) for x in starts(image.width)]
    total = np.zeros((int(quality_segmentation.config.num_labels), image.height, image.width), np.float32)
    counts = np.zeros((image.height, image.width), np.float32)
    for left, top, right, bottom in crops:
        crop = image.crop((left, top, right, bottom))
        if contract.get('input_size'):
            import cv2
            size = contract['input_size']
            crop = cv2.resize(np.asarray(crop), (size, size), interpolation=cv2.INTER_LINEAR)
        inputs = quality_segmentation_processor(images=crop, return_tensors='pt')
        values = inputs['pixel_values'].to(quality_segmentation.device, dtype=quality_segmentation.dtype)
        logits = quality_segmentation(pixel_values=values).logits.float()
        if not torch.isfinite(logits).all():
            raise ValueError('Non-finite semantic logits')
        probabilities = torch.nn.functional.interpolate(logits, size=(bottom - top, right - left), mode='bilinear', align_corners=False)[0].softmax(dim=0).cpu().numpy()
        total[:, top:bottom, left:right] += probabilities
        counts[top:bottom, left:right] += 1
    if not (counts > 0).all():
        raise ValueError('Uncovered segmentation grid')
    return total / counts[None]


In [ ]:
import shutil
audit_target = ROOT / 'runtime-audit'
if audit_target.exists(): raise RuntimeError('Audit output exists. Preserve it and use a fresh session; no output is overwritten.')


In [ ]:
"""Embedded by build-kaggle-pack.py: real inference only, never trains or changes a gate."""
import cv2
import time
from types import SimpleNamespace


def json_safe(value):
    if isinstance(value, dict):
        return {k: json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    return None if isinstance(value, float) and not np.isfinite(value) else value

assert warm_start, "Attach your extracted notebook 02 checkpoint before Run All."
audit_dir = ROOT / "runtime-audit"
audit_dir.mkdir(parents=True, exist_ok=True)
quality_segmentation = trainer.model.float().eval()
quality_segmentation_processor = processor
SEGMENTATION_PREPROCESSING = quality_preprocessing_contract(old_root)
QUALITY_TILED_SEGMENTATION = False

# Reproduce the FULL validation protocol at its original 384px label grid first.
# This is a regression/reproduction run, not a newly untouched test split.
reproduced = trainer.evaluate()
recorded = old_manifest["metrics"]
metric_names = ["eval_mean_iou", "eval_iou_water", "eval_iou_forest", "eval_iou_agricultural"]
reproduction_deltas = {key: float(reproduced[key] - recorded[key]) for key in metric_names}
reproduction_passed = all(abs(delta) <= 0.005 for delta in reproduction_deltas.values())
print({"full_validation_reproduced": reproduction_passed, "metric_deltas": reproduction_deltas})

# A fixed seed chooses scenes BEFORE inference. Both modes see exactly the same examples,
# labels, precision and output grid. The only changed variable is the image resize.
positions = np.sort(np.random.default_rng(20260924).choice(
    len(validation_dataset), size=min(200, len(validation_dataset)), replace=False
))
matrices = {mode: np.zeros((7, 7), np.int64) for mode in ("legacy_native", "training_matched")}
records = []
palette = np.array([[122,132,146], [247,166,77], [245,232,112], [44,174,235],
                    [191,134,78], [57,166,93], [157,207,89]], np.uint8)
for number, position in enumerate(tqdm(positions, desc="Runtime preprocessing comparison")):
    item = validation_source[int(position)]
    rgb = item["image"].permute(1, 2, 0).byte().numpy()
    publisher_mask = item["mask"].numpy().astype(np.uint8)
    reference = np.where(publisher_mask == 0, 255, publisher_mask - 1).astype(np.uint8)
    image = Image.fromarray(rgb)
    source_path = Path(validation_source.files[int(position)]["image"])
    scene_id = f"{source_path.parent.parent.name}/{source_path.name}"
    row = {"scene_id": scene_id, "validation_position": int(position),
           "image_sha256": hashlib.sha256(source_path.read_bytes()).hexdigest(), "modes": {}}
    previews = [image]
    for mode in matrices:
        SEGMENTATION_PREPROCESSING = (quality_preprocessing_contract(old_root)
                                     if mode == "training_matched" else {"input_size": None})
        torch.cuda.synchronize()
        started = time.perf_counter()
        probabilities = quality_semantic_prediction(image)
        torch.cuda.synchronize()
        prediction = probabilities.argmax(axis=0).astype(np.uint8)
        elapsed = time.perf_counter() - started
        confusion = np.zeros((7, 7), np.int64)
        update_confusion(confusion, prediction, reference)
        matrices[mode] += confusion
        row["modes"][mode] = {"seconds": elapsed, "confusion": confusion.tolist(),
                               "metrics": metrics_from_confusion(confusion)}
        path = audit_dir / "predictions" / mode / scene_id
        path.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(prediction).save(path.with_suffix(".png"))
        if number < 12:
            color = palette[prediction].copy()
            color[reference == 255] = rgb[reference == 255]
            previews.append(Image.blend(image, Image.fromarray(color), 0.45))
        del probabilities, prediction
    if number < 12:
        canvas = Image.new("RGB", (image.width * 3, image.height + 32), "white")
        from PIL import ImageDraw
        draw = ImageDraw.Draw(canvas)
        for index, (preview, title) in enumerate(zip(previews, ["Source", "Old native input", "Training-matched input"])):
            canvas.paste(preview, (image.width * index, 32))
            draw.text((image.width * index + 8, 8), title, fill="black")
        canvas.save(audit_dir / f"preview_{number:02d}.png")
    records.append(row)
    # Save incrementally: a session interruption must not lose the completed predictions.
    with (audit_dir / "scene_predictions.jsonl").open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(json_safe(row), allow_nan=False) + "\n")

summary = {"purpose": "preprocessing ablation; does not promote a checkpoint or calibrate confidence",
           "weights_sha256": warm_start["weights_sha256"],
           "preprocessing": quality_preprocessing_contract(old_root),
           "comparison_scenes": len(records), "precision": "float32",
           "full_validation_reproduced": reproduction_passed, "reproduction_deltas": reproduction_deltas,
           "full_validation_metrics": reproduced,
           "native_label_grid_comparison": {mode: metrics_from_confusion(cm) for mode, cm in matrices.items()},
           "bootstrap": {}, "release_candidate_unchanged": old_manifest.get("release_candidate")}
# Paired SCENE bootstrap, never treat a million correlated pixels as a million samples.
rng = np.random.default_rng(20260924)
scene_cm = {mode: np.array([r["modes"][mode]["confusion"] for r in records]) for mode in matrices}
for metric in ("mean_iou", "iou_water", "iou_forest", "iou_agricultural"):
    deltas = []
    for _ in range(1000):
        selected = rng.integers(0, len(records), size=len(records))
        old = metrics_from_confusion(scene_cm["legacy_native"][selected].sum(axis=0))[metric]
        new = metrics_from_confusion(scene_cm["training_matched"][selected].sum(axis=0))[metric]
        if np.isfinite(new - old):
            deltas.append(new - old)
    summary["bootstrap"][metric] = {"paired_scene_delta_ci95": np.quantile(deltas, [0.025, 0.975]).tolist() if deltas else None}
(audit_dir / "summary.json").write_text(json.dumps(json_safe(summary), indent=2, allow_nan=False), encoding="utf-8")
export_inference_zip(audit_dir, "/kaggle/working/00_runtime_preprocessing_evidence.zip")
print(json.dumps(summary, indent=2))
print("Preserve the ZIP. No training occurred, no weights changed, and no release gate was relaxed.")
if not reproduction_passed:
    raise RuntimeError("Recorded validation was not reproduced within 0.005. Review the saved evidence before training/serving.")
